# Lab 2-1：Adult Census Income 二分类

根据个人基本信息预测年收入是否超过 50K：`>50K` 记为 `1`，`<=50K` 记为 `0`。

本 notebook 会：

1. 自动定位竞赛的 `train.csv`、`test.csv` 和 `sample_submission.csv`；
2. 将 `?` 视为缺失值，并分别处理数值特征与类别特征；
3. 使用分层验证集评估 Accuracy，并只在验证集上选择分类阈值；
4. 用全部训练数据重新训练，生成 `/kaggle/working/submission.csv`。

> 模型只使用 scikit-learn，Kaggle CPU 环境可直接运行，不需要联网或安装额外依赖。

## 0. 导入依赖并定位竞赛数据

In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42

# Kaggle 上使用默认路径；环境变量仅用于本地复现。
INPUT_ROOT = Path(os.environ.get('LAB2_INPUT_ROOT', '/kaggle/input')).expanduser()
WORKING_DIR = Path(os.environ.get('LAB2_WORKING_DIR', '/kaggle/working')).expanduser()

assert INPUT_ROOT.exists(), f'找不到输入目录：{INPUT_ROOT}'
csv_files = sorted(
    path for path in INPUT_ROOT.rglob('*')
    if path.is_file() and path.suffix.lower() == '.csv'
)

print('Kaggle Input 中的 CSV 文件：')
for path in csv_files:
    print(' -', path)

files_by_parent = {}
for path in csv_files:
    files_by_parent.setdefault(path.parent, {})[path.name.lower()] = path

candidates = []
for parent, name_map in files_by_parent.items():
    train_path = name_map.get('train.csv')
    test_path = name_map.get('test.csv')
    if train_path is None or test_path is None:
        continue

    sample_path = next((
        path for name, path in name_map.items()
        if 'sample' in name and ('submission' in name or 'submit' in name)
    ), None)
    score = (
        int(sample_path is not None),
        int('2026' in str(parent).lower()),
        int('实验2' in str(parent) or 'lab2' in str(parent).lower()),
    )
    candidates.append((score, parent, train_path, test_path, sample_path))

if not candidates:
    raise FileNotFoundError('没有找到位于同一目录的 train.csv 和 test.csv。')

candidates.sort(key=lambda item: item[0], reverse=True)
best_score = candidates[0][0]
best_candidates = [item for item in candidates if item[0] == best_score]
if len(best_candidates) > 1:
    raise RuntimeError(
        '发现多个同等匹配的数据目录，请只挂载本竞赛数据：'
        + ', '.join(str(item[1]) for item in best_candidates)
    )

_, DATA_DIR, TRAIN_PATH, TEST_PATH, SAMPLE_PATH = best_candidates[0]
print(f'\n使用数据目录：{DATA_DIR}')
print('TRAIN_PATH :', TRAIN_PATH)
print('TEST_PATH  :', TEST_PATH)
print('SAMPLE_PATH:', SAMPLE_PATH)

## 1. 读取数据并识别标签列、ID 列

标签列应只存在于训练集；若列名是 `income`、`label`、`target` 等则优先识别，否则使用训练集比测试集多出的唯一一列。提交格式以真实的样例提交文件为准。

In [ ]:
ADULT_FEATURE_COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
]

def read_adult_csv(path, role):
    # 先按有表头方式读取；UCI 原始测试文件可能含以 | 开头的说明行。
    frame = pd.read_csv(path, skipinitialspace=True, comment='|')
    normalized_columns = {str(column).strip().lower() for column in frame.columns}
    has_named_header = {'age', 'workclass', 'education'}.issubset(normalized_columns)
    if has_named_header:
        print(f'{path.name}：检测到表头')
        return frame

    # 竞赛文件无表头时，默认读取会把第一条样本误当作列名；重新读取并补上标准列名。
    frame = pd.read_csv(
        path, header=None, skipinitialspace=True, comment='|', skip_blank_lines=True
    ).dropna(how='all').reset_index(drop=True)

    if role == 'train' and frame.shape[1] == 15:
        frame.columns = ADULT_FEATURE_COLUMNS + ['income']
    elif role == 'train' and frame.shape[1] == 16:
        frame.columns = ['id'] + ADULT_FEATURE_COLUMNS + ['income']
    elif role == 'test' and frame.shape[1] == 14:
        frame.columns = ADULT_FEATURE_COLUMNS
    elif role == 'test' and frame.shape[1] == 15:
        last_values = (
            frame.iloc[:, -1].astype('string').str.strip().str.rstrip('.').str.upper()
        )
        income_tokens = {'<=50K', '>50K', '0', '1'}
        if set(last_values.dropna().unique()).issubset(income_tokens):
            # 本竞赛下载的 test.csv 保留了 UCI 原始测试标签。
            frame.columns = ADULT_FEATURE_COLUMNS + ['income']
        else:
            first_column = pd.to_numeric(frame.iloc[:, 0], errors='coerce')
            looks_like_id = first_column.notna().all() and first_column.is_unique
            if not looks_like_id:
                raise ValueError(
                    '无表头 test.csv 有 15 列，既未识别出收入标签，也未识别出 ID。'
                )
            frame.columns = ['id'] + ADULT_FEATURE_COLUMNS
    else:
        raise ValueError(
            f'{role} 文件无表头且有 {frame.shape[1]} 列，不符合 Adult 数据格式。'
        )

    print(f'{path.name}：未检测到表头，已补充标准 Adult 列名')
    return frame

train_df = read_adult_csv(TRAIN_PATH, role='train')
test_df = read_adult_csv(TEST_PATH, role='test')
sample_df = pd.read_csv(SAMPLE_PATH) if SAMPLE_PATH is not None else None

for frame in (train_df, test_df, sample_df):
    if frame is not None:
        frame.columns = [str(column).strip() for column in frame.columns]
        if frame.columns.duplicated().any():
            raise ValueError(f'发现重复列名：{frame.columns[frame.columns.duplicated()].tolist()}')

def normalized_name(name):
    return ''.join(character for character in str(name).lower() if character.isalnum())

target_names = {'label', 'target', 'income', 'salary', 'class', 'y'}
train_only_columns = [column for column in train_df.columns if column not in test_df.columns]
explicit_targets = [
    column for column in train_df.columns
    if normalized_name(column) in target_names
]

if len(explicit_targets) == 1:
    TARGET_COLUMN = explicit_targets[0]
elif len(train_only_columns) == 1:
    TARGET_COLUMN = train_only_columns[0]
else:
    raise ValueError(
        '无法唯一识别标签列。训练集独有列为：' + str(train_only_columns)
    )

id_names = {'id', 'index', 'rowid', 'rowindex'}
id_candidates = [
    column for column in test_df.columns
    if normalized_name(column) in id_names
]
ID_COLUMN = id_candidates[0] if id_candidates else None

feature_columns = [
    column for column in test_df.columns
    if column in train_df.columns and column not in {ID_COLUMN, TARGET_COLUMN}
]
if not feature_columns:
    raise ValueError('训练集和测试集之间没有可用的共同特征列。')

print('训练集形状：', train_df.shape)
print('测试集形状：', test_df.shape)
print('标签列：', TARGET_COLUMN)
print('ID 列：', ID_COLUMN)
TEST_TARGET_AVAILABLE = TARGET_COLUMN in test_df.columns
print('测试集是否含参考标签：', TEST_TARGET_AVAILABLE)
print('特征数：', len(feature_columns))
display(train_df.head())
if sample_df is not None:
    print('样例提交形状：', sample_df.shape)
    display(sample_df.head())

## 2. 清洗缺失值并编码训练标签

`?`、空字符串和纯空白统一记为缺失值。标签兼容 `>50K` / `<=50K`、带句点的 UCI 原始测试标签，以及数值 `1` / `0`。

In [ ]:
def clean_string_columns(frame):
    cleaned = frame.copy()
    string_columns = cleaned.select_dtypes(
        include=['object', 'string', 'category']
    ).columns
    for column in string_columns:
        values = cleaned[column].astype('string').str.strip()
        cleaned[column] = values.replace(
            {'?': pd.NA, '': pd.NA, 'nan': pd.NA, 'None': pd.NA}
        )
    return cleaned

def encode_income_target(series):
    values = (
        series.astype('string')
        .str.strip()
        .str.rstrip('.')
        .str.replace(r'\s+', '', regex=True)
        .str.upper()
    )

    encoded = pd.Series(pd.NA, index=series.index, dtype='Int8')
    encoded[values.isin(['1', '>50K', 'TRUE', 'YES'])] = 1
    encoded[values.isin(['0', '<=50K', 'FALSE', 'NO'])] = 0

    if encoded.isna().any():
        unknown = sorted(values[encoded.isna()].dropna().unique().tolist())
        raise ValueError(f'标签列中存在无法识别的值：{unknown}')

    return encoded.astype('int8')

train_clean = clean_string_columns(train_df)
test_clean = clean_string_columns(test_df)

y = encode_income_target(train_clean[TARGET_COLUMN])
y_test_reference = (
    encode_income_target(test_clean[TARGET_COLUMN])
    if TEST_TARGET_AVAILABLE else None
)
X = train_clean[feature_columns].copy()
X_test = test_clean[feature_columns].copy()

# 若某列几乎全部可以解析为数值，则统一转为数值类型。
for column in feature_columns:
    combined = pd.concat([X[column], X_test[column]], ignore_index=True)
    observed = combined.notna()
    if not observed.any():
        continue

    parsed = pd.to_numeric(combined, errors='coerce')
    numeric_ratio = parsed[observed].notna().mean()
    if numeric_ratio >= 0.98:
        X[column] = pd.to_numeric(X[column], errors='coerce')
        X_test[column] = pd.to_numeric(X_test[column], errors='coerce')

categorical_features = X.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()
numeric_features = [
    column for column in feature_columns
    if column not in categorical_features
]

# 使用 np.nan 可兼容不同 scikit-learn 版本的 SimpleImputer。
for column in categorical_features:
    X[column] = X[column].astype('object').where(X[column].notna(), np.nan)
    X_test[column] = X_test[column].astype('object').where(X_test[column].notna(), np.nan)

print('数值特征：', numeric_features)
print('类别特征：', categorical_features)
print('\n标签分布：')
display(
    pd.DataFrame({
        'count': y.value_counts().sort_index(),
        'ratio': y.value_counts(normalize=True).sort_index(),
    })
)
print('训练集缺失值总数：', int(X.isna().sum().sum()))
print('测试集缺失值总数：', int(X_test.isna().sum().sum()))

## 3. 分层验证、训练模型并选择 Accuracy 阈值

预处理器只在训练折上拟合，因此验证集不会泄漏到缺失值填充或类别编码中。类别特征使用 One-Hot 编码，分类器使用 `HistGradientBoostingClassifier`。

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

try:
    one_hot = OneHotEncoder(
        handle_unknown='ignore',
        min_frequency=2,
        sparse_output=False,
        dtype=np.float32,
    )
except TypeError:
    # 兼容较旧的 scikit-learn。
    one_hot = OneHotEncoder(
        handle_unknown='ignore',
        min_frequency=2,
        sparse=False,
        dtype=np.float32,
    )

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', one_hot),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ],
    remainder='drop',
)

validation_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=300,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=30,
    random_state=RANDOM_STATE,
)

validation_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', validation_model),
])
validation_pipeline.fit(X_train, y_train)

valid_probability = validation_pipeline.predict_proba(X_valid)[:, 1]
thresholds = np.arange(0.30, 0.701, 0.005)
threshold_scores = np.array([
    accuracy_score(y_valid, (valid_probability >= threshold).astype('int8'))
    for threshold in thresholds
])
best_score = threshold_scores.max()
best_indices = np.flatnonzero(np.isclose(threshold_scores, best_score))
best_index = best_indices[np.argmin(np.abs(thresholds[best_indices] - 0.50))]
BEST_THRESHOLD = float(thresholds[best_index])
valid_prediction = (valid_probability >= BEST_THRESHOLD).astype('int8')
BEST_ITERATIONS = max(50, validation_pipeline.named_steps['model'].n_iter_)

print(f'验证集 Accuracy：{accuracy_score(y_valid, valid_prediction):.6f}')
print(f'最佳阈值：{BEST_THRESHOLD:.3f}')
print(f'早停迭代数：{BEST_ITERATIONS}')
print('\n混淆矩阵（行是真实标签，列是预测标签）：')
display(pd.DataFrame(
    confusion_matrix(y_valid, valid_prediction),
    index=['true_0', 'true_1'],
    columns=['pred_0', 'pred_1'],
))
print(classification_report(y_valid, valid_prediction, digits=4))

## 4. 使用全部训练数据重新训练并预测测试集

最终模型沿用验证阶段得到的迭代轮数，并关闭内部早停，使 32,561 条训练数据全部参与拟合。当前下载的 `test.csv` 保留了 UCI 原始收入标签；该列会从输入特征中剔除，只在模型预测完成后用于独立核对 Accuracy。

In [ ]:
final_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=BEST_ITERATIONS,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=RANDOM_STATE,
)

final_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', final_model),
])
final_pipeline.fit(X, y)

test_probability = final_pipeline.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= BEST_THRESHOLD).astype('int8')

print('测试集预测数量：', len(test_prediction))
print('预测标签取值：', sorted(np.unique(test_prediction).tolist()))
print('预测为 >50K 的比例：', f'{test_prediction.mean():.4f}')
if y_test_reference is not None:
    print(
        '本地 test.csv 参考标签 Accuracy（未参与训练或阈值选择）：',
        f'{accuracy_score(y_test_reference, test_prediction):.6f}',
    )

## 5. 按样例格式生成 `submission.csv`

若竞赛提供样例提交文件，则完整保留其 ID 列、列名和顺序，只替换唯一的预测列；否则生成 `id,label` 两列。

In [ ]:
if sample_df is not None:
    submission = sample_df.copy()
    if len(submission) != len(test_prediction):
        raise ValueError(
            f'样例提交有 {len(submission)} 行，但测试集预测有 {len(test_prediction)} 行。'
        )

    sample_id_candidates = [
        column for column in submission.columns
        if normalized_name(column) in id_names
    ]
    sample_id_column = sample_id_candidates[0] if sample_id_candidates else submission.columns[0]
    prediction_columns = [
        column for column in submission.columns
        if column != sample_id_column
    ]
    if len(prediction_columns) != 1:
        raise ValueError(
            '样例提交中应有一个 ID 列和一个预测列，实际列为：'
            + str(submission.columns.tolist())
        )
    prediction_column = prediction_columns[0]

    if ID_COLUMN is not None:
        sample_ids = submission[sample_id_column].astype('string').str.strip()
        test_ids = test_df[ID_COLUMN].astype('string').str.strip()
        if not sample_ids.reset_index(drop=True).equals(test_ids.reset_index(drop=True)):
            raise ValueError('样例提交中的 ID 顺序与 test.csv 不一致。')

    submission[prediction_column] = test_prediction
else:
    if ID_COLUMN is not None:
        id_name = ID_COLUMN
        id_values = test_df[ID_COLUMN].to_numpy()
    else:
        id_name = 'id'
        id_values = np.arange(len(test_prediction))

    prediction_column = 'label'
    submission = pd.DataFrame({
        id_name: id_values,
        prediction_column: test_prediction,
    })

assert len(submission) == len(test_df)
assert not submission.isna().any().any()
assert set(pd.to_numeric(submission[prediction_column]).unique()).issubset({0, 1})

WORKING_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = WORKING_DIR / 'submission.csv'
submission.to_csv(OUTPUT_PATH, index=False)

print('提交文件已生成：', OUTPUT_PATH)
print('提交文件形状：', submission.shape)
display(submission.head())
display(submission.tail())

运行完成后，在 Kaggle 右侧 **Output** 中找到 `submission.csv`，确认行数为 16,281、预测列只包含 `0` 和 `1`，再提交到竞赛。